<a href="https://colab.research.google.com/github/gunnsmart/science-skills/blob/arena%2F01a0bfbe-science-skills/Image_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Generator — Open-model Workbench

**เลือกโมเดล → ตั้งค่า → สร้างภาพ → ดาวน์โหลด PNG / JPEG + metadata**

Notebook แบบ standalone สำหรับ **Google Colab / Jupyter** ไม่ต้อง clone repository และไม่มี web server

1. Colab: เลือก **Runtime → Change runtime type → GPU** แล้วรันเซลล์ตามลำดับ
2. ติดตั้ง dependencies ก่อน import; ถ้าเคย import เวอร์ชันเก่าแล้ว ให้ restart runtime หลังติดตั้ง
3. เริ่มด้วย **SD 1.5** (512px) บน GPU ขนาดเล็ก หรือ **SDXL**; โมเดลใหญ่ เช่น FLUX / Qwen อาจใช้ RAM / VRAM มากเกิน Colab ฟรี แม้ offload แล้ว
4. เลือก preset หรือ **Custom** แล้วใส่ Hugging Face repo ID / local Diffusers directory / single-file `.safetensors`
5. ตั้ง `RUN_GENERATE = True` ในเซลล์สร้างภาพเมื่อพร้อม (ค่าเริ่มต้นไม่ดาวน์โหลด weights)

### ขอบเขตที่รองรับจริง
- Text-to-image ผ่าน **Diffusers pipelines ที่ติดตั้งอยู่** ไม่จำกัดเฉพาะรายชื่อ preset: repo ใหม่ / fine-tune ใช้ Custom ได้เมื่อสถาปัตยกรรมรองรับ
- Single-file checkpoint ต้องมีตัวแปลง `from_single_file` และเลือก pipeline ให้ตรงสถาปัตยกรรม; `.safetensors` **ไม่ใช่**รูปแบบที่ทำให้ทุกโมเดลโหลดร่วมกันได้
- โมเดลที่ใช้โค้ดเฉพาะ, GGUF, quantization เฉพาะทาง, multi-stage workflow, image-editing, ControlNet และ inpainting ไม่ได้รองรับโดยอัตโนมัติ ต้องเขียน local adapter (ตัวอย่างท้าย notebook)
- **ไม่รับประกัน “ทุกโมเดล Open source”**: open weights ไม่เท่ากับ open-source license หรืออนุญาตเชิงพาณิชย์ อ่าน model card/license ของแต่ละ repo ก่อนใช้; preset เป็นค่าเริ่มต้น ไม่ใช่ผลทดสอบ GPU ทุกตัว
- ไม่เปิด remote Python code และไม่โหลด pickle `.ckpt` โดยอัตโนมัติ ไม่ปิด safety checker ที่โมเดลมีให้

ภาพ, prompt, seed และ config จะอยู่ใน ZIP: อย่าแชร์ metadata หาก prompt มีข้อมูลส่วนตัว ส่วน token จะไม่ถูกบันทึก

In [ ]:
#@title 1 — Install dependencies (รันก่อน import)
INSTALL_DEPENDENCIES = True #@param {type:"boolean"}
import importlib.util
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    # Keep Colab's CUDA-compatible torch rather than force-replacing it.
    if importlib.util.find_spec("torch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch", "torchvision"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade",
        "diffusers>=0.36.0,<1", "transformers>=4.51,<5", "accelerate>=1.2,<2",
        "huggingface_hub>=0.34,<1", "safetensors>=0.4", "sentencepiece",
        "protobuf", "pillow>=10", "peft>=0.15,<1"], check=True)
    print("ติดตั้งแล้ว — ถ้าเคย import dependencies ใน runtime นี้ ให้ restart ก่อนรันเซลล์ถัดไป")

In [ ]:
#@title 2 — Environment + optional Hugging Face authentication
import gc
import os
import platform
from importlib.metadata import version
from pathlib import Path
import torch
import diffusers
from PIL import Image
from IPython.display import display, FileLink

WORK_DIR = Path("/content/ImageGenerator" if Path("/content").exists() else "image_generator_outputs").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
HAS_CUDA = torch.cuda.is_available()
print("Python:", platform.python_version())
print({name: version(name) for name in ("torch", "diffusers", "transformers", "accelerate")})
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM:", round(props.total_memory / 1024**3, 1), "GiB")
else:
    print("CPU mode: ช้ามากและโมเดลใหญ่ใช้ RAM สูง — แนะนำ GPU runtime")

# Optional: Colab Secrets named HF_TOKEN, or environment HF_TOKEN, or existing HF login.
# Never paste a token into source code / form fields or include it in saved metadata.
HF_TOKEN = os.environ.get("HF_TOKEN") or None
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF authentication:", "secret supplied" if HF_TOKEN else "cached login / public access")
print("Gated repos: ยอมรับ license และขอสิทธิ์ใน model card ด้วยบัญชีเจ้าของ token ก่อน")

In [ ]:
#@title 3 — Model registry (เพิ่ม preset ได้ ไม่ได้จำกัด Custom models)
# class names are built-in Diffusers classes, NOT downloaded Python modules.
# RAM/VRAM depends on dtype, resolution, offload, runtime and model revision.
MODEL_REGISTRY = {
    "SD 1.5": dict(repo="stable-diffusion-v1-5/stable-diffusion-v1-5", pipeline="StableDiffusionPipeline", size=512, steps=30, guidance=7.5, multiple=8, precision="FP16", note="OpenRAIL; จุดเริ่มต้นสำหรับ GPU เล็ก"),
    "SDXL": dict(repo="stabilityai/stable-diffusion-xl-base-1.0", pipeline="StableDiffusionXLPipeline", size=1024, steps=30, guidance=5.0, multiple=8, precision="FP16", note="OpenRAIL++; base pipeline ไม่รวม refiner"),
    "SDXL Turbo": dict(repo="stabilityai/sdxl-turbo", pipeline="StableDiffusionXLPipeline", size=512, steps=4, guidance=0.0, multiple=8, precision="FP16", note="ตรวจ license; distilled model, CFG=0"),
    "SD 3.5 Medium": dict(repo="stabilityai/stable-diffusion-3.5-medium", pipeline="StableDiffusion3Pipeline", size=1024, steps=28, guidance=4.5, multiple=16, precision="BF16", note="Gated / Stability Community License; RAM สูง"),
    "FLUX.1 schnell": dict(repo="black-forest-labs/FLUX.1-schnell", pipeline="FluxPipeline", size=1024, steps=4, guidance=0.0, multiple=16, precision="BF16", extra={"max_sequence_length":256}, note="Apache-2.0; โมเดลใหญ่; ไม่ใช้ negative prompt แบบ SD"),
    "FLUX.1 dev": dict(repo="black-forest-labs/FLUX.1-dev", pipeline="FluxPipeline", size=1024, steps=28, guidance=3.5, multiple=16, precision="BF16", note="Gated / non-commercial model license; ไม่ใช่ unrestricted open source"),
    "PixArt Sigma": dict(repo="PixArt-alpha/PixArt-Sigma-XL-2-1024-MS", pipeline="PixArtSigmaPipeline", size=1024, steps=20, guidance=4.5, multiple=16, precision="FP16", note="ตรวจ model card/license; T5 text encoder ใช้ RAM เพิ่ม"),
    "Sana": dict(repo="Efficient-Large-Model/Sana_1600M_1024px_diffusers", pipeline="SanaPipeline", size=1024, steps=20, guidance=5.0, multiple=32, precision="BF16", note="ตรวจ model card/license; อาจต้องสิทธิ์ text encoder"),
    "Lumina 2": dict(repo="Alpha-VLLM/Lumina-Image-2.0", pipeline="Lumina2Pipeline", size=1024, steps=30, guidance=4.0, multiple=32, precision="BF16", note="ตรวจ model card/license; ใช้ BF16 หรือ FP32"),
    "AuraFlow": dict(repo="fal/AuraFlow-v0.3", pipeline="AuraFlowPipeline", size=1024, steps=50, guidance=3.5, multiple=16, precision="FP16", note="ตรวจ model card/license; โมเดลใหญ่"),
    "HunyuanDiT": dict(repo="Tencent-Hunyuan/HunyuanDiT-Diffusers", pipeline="HunyuanDiTPipeline", size=1024, steps=50, guidance=5.0, multiple=16, precision="FP16", note="ตรวจ Tencent license; text-to-image ไม่ใช่ video"),
    "Qwen Image": dict(repo="Qwen/Qwen-Image", pipeline="QwenImagePipeline", size=1024, steps=50, guidance=4.0, guidance_parameter="true_cfg_scale", multiple=16, precision="BF16", note="Apache-2.0; โมเดลใหญ่มาก ไม่เหมาะกับ Colab RAM ต่ำ"),
    "Z-Image Turbo": dict(repo="Tongyi-MAI/Z-Image-Turbo", pipeline="ZImagePipeline", size=1024, steps=9, guidance=0.0, multiple=16, precision="BF16", note="Apache-2.0; ต้องใช้ Diffusers ที่มี ZImagePipeline"),
}

for name, preset in MODEL_REGISTRY.items():
    # Inspect the lazy module namespace without importing every optional pipeline.
    available = preset["pipeline"] in dir(diffusers)
    print(f"{name}: {'class installed' if available else 'upgrade Diffusers required'} | {preset['note']}")
    print("  Model card:", "https://huggingface.co/" + preset["repo"])
print("class installed ≠ weights downloaded / authenticated / tested on this GPU")

In [ ]:
#@title 4 — Settings (0 = preset default สำหรับขนาด/steps; -1 = preset guidance)
MODEL = "SD 1.5" #@param ["SD 1.5", "SDXL", "SDXL Turbo", "SD 3.5 Medium", "FLUX.1 schnell", "FLUX.1 dev", "PixArt Sigma", "Sana", "Lumina 2", "AuraFlow", "HunyuanDiT", "Qwen Image", "Z-Image Turbo", "Custom"]
CUSTOM_MODEL = "" #@param {type:"string"}
SOURCE = "Repository / directory" #@param ["Repository / directory", "Single safetensors file"]
PIPELINE_CLASS = "Auto" #@param {type:"string"}
# Auto uses the preset class, or model_index.json for Custom repositories.
# Custom single file: enter e.g. StableDiffusionXLPipeline explicitly.
SINGLE_FILE_CONFIG = "" #@param {type:"string"}
# Optional matching Diffusers config repo/directory for from_single_file.
REVISION = "" #@param {type:"string"}
# Pin a commit SHA for a HF repo (blank = default branch / revision in single-file URL).
# For single-file checkpoints this pins weights, not the separate config repo.
BACKEND = "diffusers" #@param {type:"string"}
PROMPT = "A tiny observatory on a green mountain, sunrise, soft clouds, detailed landscape photography" #@param {type:"string"}
NEGATIVE_PROMPT = "" #@param {type:"string"}
WIDTH = 0 #@param {type:"integer"}
HEIGHT = 0 #@param {type:"integer"}
STEPS = 0 #@param {type:"integer"}
GUIDANCE = -1.0 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}
# -1 = random base seed; each image gets base_seed + index.
NUM_IMAGES = 1 #@param {type:"slider", min:1, max:8, step:1}
PRECISION = "Auto" #@param ["Auto", "FP16", "BF16", "FP32"]
MEMORY_MODE = "Auto" #@param ["Auto", "GPU", "Model CPU offload", "Sequential CPU offload", "CPU"]
VAE_TILING = True #@param {type:"boolean"}
LORA_SOURCE = "" #@param {type:"string"}
LORA_WEIGHT_NAME = "" #@param {type:"string"}
LORA_SCALE = 1.0 #@param {type:"number"}
# Optional local/HF LoRA; must match the model architecture. Leave blank to disable.
EXTRA_KWARGS_JSON = "{}" #@param {type:"string"}
# Example for supported pipelines: {"max_sequence_length": 256}. Qwen CFG uses GUIDANCE above.
# Unknown keys and overrides of the active guidance control fail, never silently ignored.
EXPORT_JPEG = True #@param {type:"boolean"}
KEEP_MODEL_IN_MEMORY = False #@param {type:"boolean"}
# Default unloads after generation to free RAM/VRAM; weights stay in HF disk cache.

In [ ]:
#@title 5 — Validation / pipeline argument routing (ไม่มีการโหลด weights)
import inspect
import json
import math
import secrets
from urllib.parse import urlparse, unquote


def single_file_location(source):
    """Parse only supported HF URLs; other hosts must be downloaded locally first."""
    parsed = urlparse(source)
    if not parsed.scheme and not parsed.netloc:
        return None
    if parsed.scheme != "https" or parsed.netloc not in ("huggingface.co", "hf.co"):
        raise ValueError("Single-file URL ต้องเป็น HTTPS ของ huggingface.co/hf.co; แหล่งอื่นให้ดาวน์โหลดเป็น local file ก่อน")
    if parsed.query or parsed.fragment:
        raise ValueError("ใช้ URL ที่ไม่มี query/fragment/token; ยืนยันตัวตนผ่าน HF_TOKEN เท่านั้น")
    parts = parsed.path.strip("/").split("/", 4)
    if len(parts) != 5 or parts[2] not in ("resolve", "blob") or not all(parts):
        raise ValueError("ใช้ URL รูปแบบ https://huggingface.co/org/repo/resolve/revision/file.safetensors")
    return dict(repo_id="/".join(parts[:2]), revision=unquote(parts[3]), filename=unquote(parts[4]))


def resolve_settings(values):
    model = values["MODEL"]
    if model != "Custom" and model not in MODEL_REGISTRY:
        raise ValueError("Unknown preset: " + model)
    preset = MODEL_REGISTRY.get(model, {})
    source = values["CUSTOM_MODEL"].strip() if model == "Custom" else preset["repo"]
    if not source:
        raise ValueError("Custom: ใส่ Hugging Face model ID หรือ local model path")
    pipeline = values["PIPELINE_CLASS"].strip()
    if pipeline == "Auto":
        pipeline = preset.get("pipeline", "Auto")
    if values["SOURCE"] not in ("Repository / directory", "Single safetensors file"):
        raise ValueError("Unknown SOURCE")
    if values["SOURCE"] == "Single safetensors file":
        if model != "Custom":
            raise ValueError("Single file: เลือก Custom แล้วใส่ path/HTTPS URL ของไฟล์")
        parsed = urlparse(source)
        if not parsed.path.lower().endswith(".safetensors"):
            raise ValueError("รับเฉพาะ .safetensors ไม่โหลด pickle .ckpt/.bin")
        location = single_file_location(source)
        revision = values["REVISION"].strip()
        if location and revision and revision != location["revision"]:
            raise ValueError("REVISION ไม่ตรงกับ revision ใน URL; แก้ให้ตรงกันหรือเว้น REVISION ว่าง")
        if pipeline == "Auto":
            raise ValueError("Single file ต้องระบุ PIPELINE_CLASS ให้ตรงสถาปัตยกรรม")
    elif source.startswith(("http://", "https://")):
        raise ValueError("ใช้ repo ID เช่น org/model ไม่ใช่ URL หน้าเว็บ")
    if not isinstance(values["PROMPT"], str) or not values["PROMPT"].strip():
        raise ValueError("Prompt ต้องไม่ว่าง")
    for name in ("WIDTH", "HEIGHT", "STEPS", "SEED", "NUM_IMAGES"):
        if type(values[name]) is not int:
            raise ValueError(name + " ต้องเป็นจำนวนเต็ม")
    # Conservative Custom alignment; architecture-specific restrictions may be stricter.
    multiple = preset.get("multiple", 32)
    width = values["WIDTH"] or preset.get("size", 1024)
    height = values["HEIGHT"] or preset.get("size", 1024)
    if any(n < 128 or n > 4096 or n % multiple for n in (width, height)):
        raise ValueError(f"ขนาดต้องอยู่ในช่วง 128–4096 และหาร {multiple} ลงตัว")
    steps = values["STEPS"] or preset.get("steps", 30)
    if not 1 <= steps <= 200:
        raise ValueError("Steps ต้องอยู่ในช่วง 1–200")
    if not 1 <= values["NUM_IMAGES"] <= 8:
        raise ValueError("NUM_IMAGES ต้องอยู่ในช่วง 1–8")
    seed = values["SEED"]
    if seed != -1 and not 0 <= seed <= 2**32 - values["NUM_IMAGES"]:
        raise ValueError("Seed ต้องเป็น -1 หรืออยู่ในช่วง uint32 ที่เพิ่มตามจำนวนภาพได้")
    if seed == -1:
        seed = secrets.randbelow(2**32 - values["NUM_IMAGES"] + 1)
    guidance = values["GUIDANCE"]
    if guidance == -1:
        guidance = preset.get("guidance", 5.0)
    if not math.isfinite(guidance) or not 0 <= guidance <= 30:
        raise ValueError("Guidance ต้องอยู่ในช่วง 0–30 หรือ -1 เพื่อใช้ preset")
    if not math.isfinite(values["LORA_SCALE"]) or not 0 <= values["LORA_SCALE"] <= 2:
        raise ValueError("LoRA scale ต้องอยู่ในช่วง 0–2")
    if values["PRECISION"] not in ("Auto", "FP16", "BF16", "FP32"):
        raise ValueError("Unknown PRECISION")
    if values["MEMORY_MODE"] not in ("Auto", "GPU", "Model CPU offload", "Sequential CPU offload", "CPU"):
        raise ValueError("Unknown MEMORY_MODE")
    extras = json.loads(values["EXTRA_KWARGS_JSON"], parse_constant=lambda x: (_ for _ in ()).throw(ValueError("Non-finite JSON: " + x)))
    json.dumps(extras, allow_nan=False)  # Also reject overflow such as 1e999.
    if not isinstance(extras, dict):
        raise ValueError("EXTRA_KWARGS_JSON ต้องเป็น JSON object")
    reserved = {"prompt", "negative_prompt", "width", "height", "num_inference_steps", "guidance_scale",
                "generator", "num_images_per_prompt", "output_type", "return_dict", "image", "mask_image"}
    if reserved.intersection(extras):
        raise ValueError("Extra kwargs ห้ามทับค่าหลัก: " + str(sorted(reserved.intersection(extras))))
    return dict(model=model, source=source, source_type=values["SOURCE"], pipeline=pipeline,
        revision=values["REVISION"].strip() or None, single_file_config=values["SINGLE_FILE_CONFIG"].strip(),
        backend=values["BACKEND"].strip(), prompt=values["PROMPT"], negative_prompt=values["NEGATIVE_PROMPT"],
        width=width, height=height, steps=steps, guidance=guidance,
        guidance_parameter=preset.get("guidance_parameter", "Auto"), seed=seed, count=values["NUM_IMAGES"],
        precision=values["PRECISION"], preferred_precision=preset.get("precision", "BF16"),
        memory_mode=values["MEMORY_MODE"], vae_tiling=values["VAE_TILING"],
        lora_source=values["LORA_SOURCE"].strip(), lora_weight_name=values["LORA_WEIGHT_NAME"].strip(),
        lora_scale=values["LORA_SCALE"], extra={**preset.get("extra", {}), **extras},
        export_jpeg=values["EXPORT_JPEG"], keep_model=values["KEEP_MODEL_IN_MEMORY"])


def build_call_kwargs(pipe, settings, generator):
    parameters = inspect.signature(pipe.__call__).parameters
    required = {"prompt", "width", "height", "num_inference_steps", "generator"}
    if required.difference(parameters):
        raise ValueError("Pipeline ไม่ตรง text-to-image contract; ต้องใช้ local adapter: " +
                         str(sorted(required.difference(parameters))))
    kwargs = dict(prompt=settings["prompt"], width=settings["width"], height=settings["height"],
                  num_inference_steps=settings["steps"], generator=generator)
    notes = []
    guidance_parameter = settings["guidance_parameter"]
    if guidance_parameter == "Auto":
        # Qwen's base model ignores guidance_scale; CFG is true_cfg_scale.
        # Keep Flux's distilled guidance separate: it also exposes both parameters.
        guidance_parameter = "true_cfg_scale" if type(pipe).__name__ == "QwenImagePipeline" else "guidance_scale"
    if guidance_parameter in parameters:
        kwargs[guidance_parameter] = settings["guidance"]
    else:
        notes.append(f"Pipeline ไม่มี {guidance_parameter}; ไม่ส่งค่านี้")
    if "negative_prompt" in parameters:
        # Empty string is intentional: some true-CFG models need an unconditional prompt.
        kwargs["negative_prompt"] = settings["negative_prompt"]
    elif settings["negative_prompt"]:
        notes.append("Pipeline ไม่มี negative_prompt; ไม่ส่งค่านี้")
    for key, value in settings["extra"].items():
        if key == guidance_parameter:
            raise ValueError(f"ใช้ช่อง GUIDANCE แทน extra kwarg {key} เพื่อไม่ให้ค่าหลักถูกทับ")
        if key not in parameters:
            raise ValueError(f"Pipeline ไม่รองรับ extra kwarg: {key}")
        kwargs[key] = value
    # One image per call avoids a multi-image VRAM spike.
    if "num_images_per_prompt" in parameters:
        kwargs["num_images_per_prompt"] = 1
    if "output_type" in parameters:
        kwargs["output_type"] = "pil"
    if "return_dict" in parameters:
        kwargs["return_dict"] = True
    return kwargs, notes

In [ ]:
#@title 6 — Model loader + memory management + optional LoRA
# Extension point: audited local Python only. See adapter instructions below.
CUSTOM_LOADERS = globals().get("CUSTOM_LOADERS", {})
PIPE = globals().get("PIPE", None)
PIPE_KEY = globals().get("PIPE_KEY", None)


def unload_model():
    global PIPE, PIPE_KEY
    PIPE = None
    PIPE_KEY = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def select_runtime(settings):
    mode = settings["memory_mode"]
    cuda = torch.cuda.is_available()
    if mode == "Auto":
        mode = "Model CPU offload" if cuda else "CPU"
    if mode != "CPU" and not cuda:
        raise ValueError("โหมดนี้ต้องมี CUDA GPU; เลือก GPU runtime หรือ CPU")
    precision = settings["precision"]
    bf16 = cuda and torch.cuda.is_bf16_supported()
    if precision == "Auto":
        precision = "FP32" if mode == "CPU" else settings["preferred_precision"]
        if precision == "BF16" and not bf16:
            precision = "FP32"
            print("GPU ไม่รองรับ BF16: ใช้ FP32 เพื่อเลี่ยง FP16 overflow; RAM/VRAM จะสูงขึ้น")
    if mode == "CPU" and precision != "FP32":
        raise ValueError("CPU mode ใน notebook นี้ใช้ FP32; เลือก Auto หรือ FP32")
    if precision == "BF16" and not bf16:
        raise ValueError("GPU ไม่รองรับ BF16; ใช้ Auto/FP32 หรือ GPU รุ่นที่รองรับ")
    return mode, {"FP16":torch.float16, "BF16":torch.bfloat16, "FP32":torch.float32}[precision]


def load_diffusers(settings, dtype):
    name = settings["pipeline"]
    cls = diffusers.DiffusionPipeline if name == "Auto" else getattr(diffusers, name, None)
    if not isinstance(cls, type) or not issubclass(cls, diffusers.DiffusionPipeline):
        raise ValueError(f"ไม่มี built-in Diffusers pipeline {name}; ตรวจชื่อ/อัปเดต Diffusers แล้ว restart runtime")
    options = dict(torch_dtype=dtype, token=HF_TOKEN)
    if settings["source_type"] == "Single safetensors file":
        if not hasattr(cls, "from_single_file"):
            raise ValueError(name + " ไม่มี from_single_file; ใช้ Diffusers repo/directory หรือ local adapter")
        if settings["single_file_config"]:
            options["config"] = settings["single_file_config"]
        source = settings["source"]
        location = single_file_location(source)
        if location:
            # Diffusers 0.36's own URL parser does not reliably handle resolve/ref URLs.
            # Download explicitly so nested paths and non-main revisions are honored.
            from huggingface_hub import hf_hub_download
            source = hf_hub_download(**location, token=HF_TOKEN)
        return cls.from_single_file(source, **options)
    return cls.from_pretrained(settings["source"], revision=settings["revision"],
                               use_safetensors=True, trust_remote_code=False, **options)


def get_pipeline(settings):
    global PIPE, PIPE_KEY
    mode, dtype = select_runtime(settings)
    # Include all loader-affecting settings, not prompt/seed. Authentication is never serialized.
    fields = ("source", "source_type", "pipeline", "revision", "single_file_config", "backend",
              "vae_tiling", "lora_source", "lora_weight_name", "lora_scale")
    loader = load_diffusers if settings["backend"] == "diffusers" else CUSTOM_LOADERS.get(settings["backend"])
    if loader is None:
        raise ValueError("Backend ยังไม่ลงทะเบียนใน CUSTOM_LOADERS: " + settings["backend"])
    # Custom loaders can consume ANY setting. Reusing only the Diffusers subset
    # could silently return a model loaded with an outdated custom configuration.
    loader_settings = {k:settings[k] for k in fields} if settings["backend"] == "diffusers" else settings
    key = (json.dumps(loader_settings, sort_keys=True), mode, str(dtype), id(loader))
    if PIPE is not None and key == PIPE_KEY:
        return PIPE, mode, str(dtype)
    unload_model()
    try:
        PIPE = loader(settings, dtype)
        if settings["lora_source"]:
            if not all(hasattr(PIPE, name) for name in ("load_lora_weights", "set_adapters")):
                raise ValueError("Pipeline นี้ไม่มี LoRA adapter API")
            opts = dict(adapter_name="user_lora", token=HF_TOKEN, use_safetensors=True)
            if settings["lora_weight_name"]:
                if not settings["lora_weight_name"].lower().endswith(".safetensors"):
                    raise ValueError("LoRA weight file ต้องเป็น .safetensors")
                opts["weight_name"] = settings["lora_weight_name"]
            PIPE.load_lora_weights(settings["lora_source"], **opts)
            PIPE.set_adapters(["user_lora"], adapter_weights=[settings["lora_scale"]])
        if settings["vae_tiling"]:
            if hasattr(PIPE, "enable_vae_tiling"):
                PIPE.enable_vae_tiling()
            else:
                print("Pipeline ไม่มี VAE tiling API; ข้าม (ไม่ได้ลด transformer memory)")
        if mode in ("GPU", "CPU"):
            PIPE.to("cuda" if mode == "GPU" else "cpu")
        else:
            method = "enable_model_cpu_offload" if mode == "Model CPU offload" else "enable_sequential_cpu_offload"
            if not hasattr(PIPE, method):
                raise ValueError(f"Pipeline ไม่รองรับ {mode}; เลือก GPU/CPU หรือใช้ adapter")
            getattr(PIPE, method)()
        PIPE_KEY = key
        return PIPE, mode, str(dtype)
    except Exception:
        unload_model()
        raise

### เพิ่ม backend สำหรับโมเดลนอก Diffusers (ขั้นสูง)

ไม่มี generic loader ที่อ่านได้ทุกสถาปัตยกรรม: ต้องติดตั้ง dependencies ตาม official repository และเขียน wrapper ให้ตรง API ก่อน ไม่ดาวน์โหลด/รันโค้ดจาก repo ที่ไม่รู้จักอัตโนมัติ

เพิ่ม code cell **ก่อนเซลล์ Generate** แล้วลงทะเบียน:
```python
# def load_my_backend(settings, dtype):
#     # Load real weights with the model's official implementation.
#     # Return your wrapper; do not return a placeholder image.
#     return MyTextToImageWrapper(...)
# CUSTOM_LOADERS["my_backend"] = load_my_backend
```
จากนั้นเลือก `MODEL = "Custom"`, ใส่ source และ `BACKEND = "my_backend"`.
Wrapper ต้องมี `.to(device)` และ `__call__(prompt, width, height, num_inference_steps, generator, ...)` แบบ explicit signature; คืน object ที่มี `.images` เป็น list ของ PIL images. ถ้ามี safety flags ให้คืน `.nsfw_content_detected` / `.unsafe_images` ด้วย กรณีไม่มี offload API ให้เลือก `GPU` หรือ `CPU`. แปลง seed/settings ไปยัง API จริงของโมเดลภายใน wrapper; **การลงทะเบียนชื่ออย่างเดียวไม่ได้ทำให้โมเดลรองรับ**.

สำหรับ Custom Diffusers repo ปกติ ไม่ต้องเขียน adapter — ใช้ `BACKEND = "diffusers"`, `PIPELINE_CLASS = "Auto"` ได้เลย.

In [ ]:
#@title 7 — Generation + export engine
from datetime import datetime, timezone
import hashlib
import zipfile


def generate_images(settings):
    # Reset links first: failed runs must not offer an old archive as a new result.
    global LAST_RUN_DIR, LAST_ZIP
    LAST_RUN_DIR = LAST_ZIP = None
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + secrets.token_hex(3)
    run_dir = WORK_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    pipe = result = None
    archive = WORK_DIR / (run_id + ".zip")
    temporary_archive = archive.with_suffix(".zip.part")
    complete = False
    try:
        pipe, mode, dtype = get_pipeline(settings)
        metadata = dict(settings=settings, runtime=dict(mode=mode, dtype=dtype,
            pipeline_class=type(pipe).__name__, python=platform.python_version(),
            packages={name:version(name) for name in ("torch", "diffusers", "transformers", "accelerate")}),
            created_at=datetime.now(timezone.utc).isoformat(), images=[], warnings=[], status="running")
        for index in range(settings["count"]):
            image_seed = settings["seed"] + index
            # CPU generator is supported by Diffusers randn_tensor and CPU offload.
            generator = torch.Generator(device="cpu").manual_seed(image_seed)
            kwargs, notes = build_call_kwargs(pipe, settings, generator)
            metadata["warnings"] = sorted(set(metadata["warnings"] + notes))
            for note in notes:
                print("⚠", note)
            print(f"Generating {index+1}/{settings['count']} | seed={image_seed}")
            with torch.inference_mode():
                result = pipe(**kwargs)
            images = getattr(result, "images", None)
            if images is None or len(images) != 1 or not isinstance(images[0], Image.Image):
                raise RuntimeError("Pipeline ต้องคืน .images เป็น list ของ PIL image จำนวน 1 ภาพ; ใช้ adapter สำหรับ API อื่น")
            for flag in ("nsfw_content_detected", "unsafe_images"):
                flags = getattr(result, flag, None)
                if flags is not None and any(flags):
                    raise RuntimeError("Pipeline safety checker flagged the output; ไม่ export ภาพนี้")
            image = images[0]
            filename = f"image_{index+1:03d}_seed_{image_seed}.png"
            image.save(run_dir / filename)
            record = dict(file=filename, seed=image_seed, width=image.width, height=image.height,
                          sha256=hashlib.sha256((run_dir / filename).read_bytes()).hexdigest())
            if settings["export_jpeg"]:
                # Flatten alpha onto white; JPEG cannot preserve transparency.
                rgba = image.convert("RGBA")
                rgb = Image.new("RGB", rgba.size, "white")
                rgb.paste(rgba, mask=rgba.getchannel("A"))
                jpg_name = filename.removesuffix(".png") + ".jpg"
                rgb.save(run_dir / jpg_name, quality=95, subsampling=0)
                record["jpeg"] = jpg_name
            metadata["images"].append(record)
            # Preserve completed-image metadata if a later image fails.
            (run_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
            display(image)
            result = None
        metadata["status"] = "complete"
        (run_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        (run_dir / "README.txt").write_text(
            "AI-generated images. Check the source model and LoRA licenses before use.\n"
            "metadata.json contains prompts/settings but no authentication tokens.\n"
            "Seeds improve repeatability, not bitwise determinism across hardware/library versions.\n",
            encoding="utf-8")
        # Publish only a closed, complete archive. Interrupted writes must not leave
        # a plausible-looking .zip that the user could mistake for a successful run.
        with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
            for file in sorted(run_dir.iterdir()):
                bundle.write(file, arcname=file.name)
        temporary_archive.replace(archive)
        LAST_RUN_DIR, LAST_ZIP = run_dir, archive
        complete = True
        print("Saved:", run_dir, "\nZIP:", archive)
        return archive
    except (Exception, KeyboardInterrupt):
        # Avoid serializing exceptions (which could contain private paths/URLs).
        metadata_path = run_dir / "metadata.json"
        if metadata_path.exists():
            partial = json.loads(metadata_path.read_text(encoding="utf-8"))
            partial["status"] = "failed_partial"
            metadata_path.write_text(json.dumps(partial, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Run ไม่สำเร็จ; ไม่มี ZIP ใหม่ ภาพที่เสร็จแล้ว (ถ้ามี) อยู่ที่", run_dir)
        raise
    finally:
        pipe = result = None
        if not settings["keep_model"] or not complete:
            unload_model()
        if not complete:
            # Best effort: filesystem errors must not mask the original failure.
            for partial_archive in (temporary_archive, archive):
                try:
                    partial_archive.unlink(missing_ok=True)
                except OSError:
                    print("ลบ archive ที่ไม่สมบูรณ์ไม่ได้:", partial_archive)

In [ ]:
#@title 8 — Generate (เปิดสวิตช์เมื่อพร้อมโหลด weights)
RUN_GENERATE = False #@param {type:"boolean"}
LAST_RUN_DIR = LAST_ZIP = None
if RUN_GENERATE:
    try:
        settings = resolve_settings(globals())
        print("Model:", settings["source"], "| Pipeline:", settings["pipeline"])
        print("Size:", settings["width"], "x", settings["height"], "| Base seed:", settings["seed"])
        generate_images(settings)
    except torch.cuda.OutOfMemoryError:
        unload_model()
        print("GPU OOM: ลด resolution / เลือก Sequential CPU offload / ใช้โมเดลเล็กลง แล้วรันใหม่")
        print("CPU offload ยังต้องใช้ system RAM; ไม่สามารถทำให้ทุกโมเดลพอดีกับ Colab ฟรีได้")
        raise
    except Exception:
        print("ตรวจ traceback: 401/403 → สิทธิ์ gated repo + HF_TOKEN; 404 → model ID/revision")
        print("Class/import error → ติดตั้ง dependencies แล้ว restart; unsupported architecture → ใช้ local adapter")
        print("LoRA mismatch → ตรวจ base architecture; missing safetensors → ใช้ checkpoint ที่ปลอดภัยและรองรับ")
        raise
else:
    print("ยังไม่โหลดโมเดล — ตั้ง RUN_GENERATE = True แล้วรันเซลล์นี้")

In [ ]:
#@title 9 — Download ZIP (PNG + optional JPEG + metadata)
DOWNLOAD_ZIP = True #@param {type:"boolean"}
if LAST_ZIP is None or not LAST_ZIP.exists():
    print("ยังไม่มี ZIP จากการ generate ที่สำเร็จในรอบนี้")
elif DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        # FileLink works when output is beneath the Jupyter notebook directory.
        display(FileLink(os.path.relpath(LAST_ZIP, Path.cwd())))
    else:
        files.download(str(LAST_ZIP))
else:
    print("ZIP:", LAST_ZIP)

In [ ]:
#@title 10 — Optional: unload weights from RAM / VRAM (ไม่ลบ HF disk cache)
UNLOAD_NOW = False #@param {type:"boolean"}
if UNLOAD_NOW:
    unload_model()
    print("Released model references; HF disk cache retained")

## Tips / troubleshooting

- **Custom model:** ใช้ repo ที่มี `model_index.json` และ safetensors; ถ้าเป็น fine-tune ของ SDXL ให้ใช้ค่า SDXL เป็นแนวทาง (1024px / 30 steps / guidance 5) ไม่ใช้ค่าของ Turbo โดยอัตโนมัติ
- **Single-file URL:** รองรับเฉพาะ Hugging Face `https://huggingface.co/org/repo/resolve/revision/file.safetensors` (หรือ `blob`) ไม่มี query/token ใน URL; revision ใน URL จะถูกใช้จริง แหล่งอื่นให้ดาวน์โหลดลง Colab แล้วใส่ local path ก่อน ส่วน `SINGLE_FILE_CONFIG` ใช้ revision เริ่มต้นของ config repo — หากต้องการ pin config ให้ดาวน์โหลด config เป็น local directory
- **Qwen Image guidance:** ช่อง `GUIDANCE` ควบคุม `true_cfg_scale` โดยตรง (default 4) ไม่ส่ง `guidance_scale` ที่ base model ไม่ได้ใช้; ห้ามตั้งค่าเดียวกันซ้ำใน Extra kwargs
- **Single-file:** ตั้ง Custom + Single safetensors file + `StableDiffusionPipeline` หรือ `StableDiffusionXLPipeline` ตาม checkpoint; กรณีตรวจ config ไม่ได้ ระบุ `SINGLE_FILE_CONFIG` เป็น Diffusers config repo ที่ตรงกัน ต้องมีอินเทอร์เน็ตสำหรับ tokenizer/config ที่ยังไม่อยู่ใน cache
- **LoRA:** ต้องเป็นสถาปัตยกรรมเดียวกับ base model และอ่าน license ทั้งสองตัว; รองรับหนึ่ง adapter ต่อรอบ ไม่โหลด pickle weights
- **Negative prompt:** ส่งเฉพาะ pipeline ที่ประกาศ parameter นี้; หากไม่รองรับจะแจ้งและจดไว้ใน metadata บางโมเดลยังต้องตั้ง CFG ตามคู่มือจึงจะมีผล
- **FP16 / BF16:** T4 ไม่รองรับ BF16; preset ที่ต้องการ BF16 จะใช้ FP32 ใน Auto บน T4 ซึ่งอาจกิน RAM มาก แนะนำเริ่ม SD 1.5 / SDXL แทนโมเดลใหญ่มาก
- **Offload:** Model CPU offload เร็วกว่า sequential แต่ต้องพอสำหรับ component ใหญ่ที่สุด ส่วน sequential ช้ากว่าและยังใช้ RAM มาก; VAE tiling ลดเฉพาะส่วน VAE ไม่มีการแอบเปลี่ยนโมเดล/ขนาดภาพเมื่อ OOM
- **Disk:** weights อาจกินพื้นที่หลายสิบ GB เก็บใน Hugging Face cache ไม่อยู่ใน ZIP หรือ Git; Colab runtime reset จะลบไฟล์ชั่วคราว ให้ดาวน์โหลดก่อน
- **Reproducibility:** ตั้ง revision เป็น commit SHA และเก็บ metadata; seed เดิมไม่รับประกันภาพเหมือนกันข้าม GPU / library versions / mutable LoRA repo
- **Safety/license:** ไม่มีการรับรองสิทธิ์เชิงพาณิชย์หรือการผ่าน stock platform; อย่าใช้ชื่อ preset แทนการอ่าน license จริง

### Verification scope
Notebook includes offline tests for validation, argument routing, loader configuration and export orchestration. A successful syntax/unit test does **not** verify model downloads or real GPU inference; each model must be smoke-tested on a suitable runtime with its actual access permissions.